In [12]:
from google.colab import files
uploaded = files.upload()

Saving vapour_pressure_60_values.csv to vapour_pressure_60_values (3).csv


In [13]:
import numpy as np
import pandas as pd

df = pd.read_csv("vapour_pressure_60_values.csv", encoding="latin-1")
raw_columns = list(df.columns)
df.columns = ["obs", "T_C", "T_K", "P_ref", "P_sim", "sigma", "split"]
df.head()

,obs,T_C,T_K,P_ref,P_sim,sigma,split
0,1,20.000000,293.150000,2.329575,2.352188,0.070566,train
1,2,21.355932,294.505932,2.532911,2.676364,0.080291,train
2,3,22.711864,295.861864,2.751555,2.762818,0.082885,train
3,4,24.067797,297.217797,2.986468,2.948478,0.088454,train
4,5,25.423729,298.573729,3.238656,3.247682,0.097430,train


In [14]:
failed = []
total = 0

def check(name, ok):
    global total
    total = total + 1
    if ok:
        print("PASSED:", name)
    else:
        print("FAILED:", name)
        failed.append(name)

R = 8.314462618
noise = df.P_sim / df.P_ref - 1
slope, intercept = np.polyfit(1 / df.T_K, np.log(df.P_ref), 1)
L = -slope * R / 1000

# temperature
check("temperature runs from 20 to 100 C", abs(df.T_C.min() - 20) < 1e-5 and abs(df.T_C.max() - 100) < 1e-5)
check("temperature steps are equal", np.diff(df.T_C).max() - np.diff(df.T_C).min() < 1e-5)
check("K = C + 273.15", np.abs(df.T_K - (df.T_C + 273.15)).max() < 1e-5)

# pressure and uncertainty
check("all values positive", (df.T_K > 0).all() and (df.P_ref > 0).all() and (df.P_sim > 0).all() and (df.sigma > 0).all())
check("uncertainty = 3% of simulated pressure", np.abs(df.sigma - 0.03 * df.P_sim).max() < 1e-5)
check("reference pressure increases with T", (np.diff(df.P_ref) > 0).all())
check("ln(P) vs 1/T is a straight line (r < -0.99)", np.corrcoef(1 / df.T_K, np.log(df.P_sim))[0, 1] < -0.99)

# noise
check("noise is about 3% (rms between 1.5% and 6%)", 0.015 < np.sqrt(np.mean(noise**2)) < 0.06)
check("no outliers (all within 15%)", np.abs(noise).max() < 0.15)

# physics
check("pressure at 100 C is about 101.325 kPa", abs(df.P_ref.iloc[-1] - 101.325) / 101.325 < 0.01)
check("latent heat is between 38 and 46 kJ/mol", 38 < L < 46)

# train / test split
check("labels are only train and test", set(df.split) == {"train", "test"})
check("36 train rows and 24 test rows", (df.split == "train").sum() == 36 and (df.split == "test").sum() == 24)
check("train rows come before test rows", df.T_K[df.split == "train"].max() < df.T_K[df.split == "test"].min())

print()
print(total - len(failed), "out of", total, "checks passed")
if len(failed) > 0:
    print("Failed checks:", failed)

PASSED: temperature runs from 20 to 100 C
PASSED: temperature steps are equal
PASSED: K = C + 273.15
PASSED: all values positive
PASSED: uncertainty = 3% of simulated pressure
PASSED: reference pressure increases with T
PASSED: ln(P) vs 1/T is a straight line (r < -0.99)
PASSED: noise is about 3% (rms between 1.5% and 6%)
PASSED: no outliers (all within 15%)
PASSED: pressure at 100 C is about 101.325 kPa
PASSED: latent heat is between 38 and 46 kJ/mol
PASSED: labels are only train and test
PASSED: 36 train rows and 24 test rows
PASSED: train rows come before test rows

14 out of 14 checks passed
